# Colab Bridge 0.2.0
连接当前运行时，上传资源状态，并按你的选择启用任务执行。
需要 GPU 时，先在 Colab 的“更改运行时类型”中选择可用 GPU。
运行下面的单元格，输入 Agent 地址与密钥；密钥输入不会显示。
模型功能的可选依赖见仓库部署说明。停止 Agent：`stop_colab_bridge()`。


In [ ]:
# Colab Bridge 0.2.0; rerun this cell to replace a stopped Agent.
import base64, getpass, hashlib, os, re, subprocess, sys, tempfile, threading
from pathlib import Path
from urllib.parse import urlsplit

_previous_stop = globals().get('_colab_bridge_stop')
_previous_thread = globals().get('_colab_bridge_thread')
if _previous_stop is not None:
    _previous_stop.set()
if _previous_thread is not None and _previous_thread.is_alive():
    _previous_thread.join(timeout=20)
    if _previous_thread.is_alive():
        raise RuntimeError('Previous Agent is stopping; wait and rerun this cell.')

_wheel_bytes = base64.b64decode('...', validate=True)
if hashlib.sha256(_wheel_bytes).hexdigest() != 'e0f8efc4063412ec99166aa953b0ded98acab3decd3c9505582dde7e12fb61a8':
    raise RuntimeError('Embedded Agent checksum mismatch')
_wheel_path = Path(tempfile.mkdtemp(prefix='colab-bridge-install-')) / 'colab_bridge_agent-0.2.0-py3-none-any.whl'
_wheel_path.write_bytes(_wheel_bytes)
def _colab_bridge_install_env(source):
    from urllib.parse import unquote, urlsplit
    allowed = {
        'PATH', 'HOME', 'TMPDIR', 'TMP', 'TEMP', 'LANG', 'LC_ALL',
        'SYSTEMROOT', 'WINDIR', 'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY',
        'SSL_CERT_FILE', 'SSL_CERT_DIR', 'REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE',
        'PIP_INDEX_URL', 'PIP_EXTRA_INDEX_URL', 'PIP_FIND_LINKS', 'PIP_CERT',
    }
    url_options = {'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY',
                   'PIP_INDEX_URL', 'PIP_EXTRA_INDEX_URL', 'PIP_FIND_LINKS'}
    clean = {}
    for name, value in source.items():
        upper = name.upper()
        if upper not in allowed or not isinstance(value, str):
            continue
        safe = True
        for part in value.split():
            decoded = unquote(part)
            if '://' not in decoded and upper not in url_options:
                continue
            # Proxy URLs may omit a scheme. Local find-links paths remain local paths.
            if upper == 'PIP_FIND_LINKS' and '://' not in decoded and '@' not in decoded:
                continue
            try:
                parsed = urlsplit(decoded if '://' in decoded else '//' + decoded)
                if (parsed.username is not None or parsed.password is not None
                        or parsed.query or parsed.fragment or not parsed.hostname):
                    safe = False
                    break
            except ValueError:
                safe = False
                break
        if safe:
            clean[name] = value
    return clean

_install_env = _colab_bridge_install_env(os.environ)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--disable-pip-version-check', 'httpx==0.28.1'], check=True, env=_install_env)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--disable-pip-version-check', '--no-deps', '--force-reinstall', str(_wheel_path)], check=True, env=_install_env)
for _module_name in list(sys.modules):
    if _module_name == 'colab_bridge_agent' or _module_name.startswith('colab_bridge_agent.'):
        del sys.modules[_module_name]

from colab_bridge_agent import __version__
from colab_bridge_agent.config import AgentConfig
from colab_bridge_agent.main import run_agent
if __version__ != '0.2.0':
    raise RuntimeError('Installed Agent version mismatch')

_agent_url = os.environ.get('COLAB_BRIDGE_AGENT_URL') or input('Colab Bridge Agent URL: ').strip()
_parsed_url = urlsplit(_agent_url)
if (_parsed_url.scheme != 'https' or not _parsed_url.hostname or _parsed_url.username
        or _parsed_url.password or _parsed_url.query or _parsed_url.fragment):
    raise ValueError('Agent URL must be HTTPS without credentials, query or fragment')
_agent_key = os.environ.get('COLAB_BRIDGE_AGENT_KEY') or getpass.getpass('Colab Bridge Agent key: ')
_runtime_label = input('Runtime label [my-colab]: ').strip() or 'my-colab'
if len(_runtime_label) > 80 or any(ord(c) < 32 for c in _runtime_label):
    raise ValueError('Runtime label must contain at most 80 printable characters')
_runtime_id = os.environ.get('COLAB_BRIDGE_RUNTIME_ID') or None
_enable_execution = os.environ.get('COLAB_BRIDGE_EXECUTION_ENABLED') == '1'
if not _enable_execution:
    _enable_execution = input('Allow your control connection to run jobs? [y/N]: ').strip().lower() in ('y', 'yes')

# Remove optional environment credentials after configuration has been read.
os.environ.pop('COLAB_BRIDGE_AGENT_KEY', None)
_colab_bridge_stop = threading.Event()
_colab_bridge_config = AgentConfig(agent_url=_agent_url, agent_key=_agent_key,
                                  label=_runtime_label, runtime_id=_runtime_id, execution_enabled=_enable_execution)
_colab_bridge_thread = threading.Thread(target=run_agent,
    args=(_colab_bridge_config,), kwargs={'stop_event': _colab_bridge_stop}, daemon=True)
_colab_bridge_thread.start()

def stop_colab_bridge():
    _colab_bridge_stop.set()
    _colab_bridge_thread.join(timeout=20)
    print('Agent stopping.' if _colab_bridge_thread.is_alive() else 'Agent stopped.')

print('Colab Bridge Agent started in the background.')
print('Job execution enabled.' if _enable_execution else 'Telemetry enabled.')
print('Run stop_colab_bridge() to stop the Agent.')
